# 📗 데이터베이스 설계와 ERD — 그리고 첫 표 만들기

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지금까지 다룬 자료는 CSV 파일과 문서 뭉치였습니다. 그런데 서비스가 실제로 굴러가면 **고객·주문·상품** 같은 기록이 매일 쌓이고, 여러 사람이 동시에 그것을 읽고 씁니다. 그 기록을 안전하게 담는 창고가 **데이터베이스**입니다.

이번 시간엔 창고의 **설계도**를 그리는 법(표·타입·제약·키·관계·ERD)을 배우고, 그 설계도를 **sqlite** 로 진짜 표로 만들어 데이터를 넣는 데까지 갑니다. sqlite 는 파이썬에 이미 들어 있어서 설치도 가입도 필요 없습니다 — **파일 하나가 곧 데이터베이스**입니다.

**오늘 이 노트북을 마치면**

- **sqlite** 파일 하나로 데이터베이스를 열고, `CREATE TABLE` · `INSERT` 로 표와 데이터를 만들 수 있습니다.
- 열마다 알맞은 **타입**과 **제약조건**을 골라 표를 설계할 수 있습니다.
- **기본키·외래키**로 두 표를 잇고, 그 관계를 **ERD** 한 장으로 읽고 그릴 수 있습니다.
- SQL 을 적는 **표기 관례**(키워드 대문자·이름 소문자)에 맞춰 문장을 쓸 수 있습니다.

## ⏪ 복습 — 지난 시간까지, 그리고 오늘

- 문서를 **임베딩**해 벡터로 만들고, **벡터 DB**에 넣어 뜻이 비슷한 것을 찾았습니다.
- 찾아온 근거로 답을 쓰는 **RAG 파이프라인**을 이어 봤습니다.

그때 다룬 것은 줄글로 된 **비정형 데이터**였습니다. 그런데 서비스에는 그것 말고도 **정형 데이터**가 있습니다 — 고객 번호, 주문 금액, 주문 날짜처럼 칸이 정해진 기록이죠. 이런 자료는 벡터로 바꿔 넣는 것이 아니라 **표에 그대로 담아** 조건으로 찾습니다.

**왜 파일이나 딕셔너리로는 부족할까요?** 슬라이드 교안에서 본 세 가지 벽입니다 — **정합성**(같은 주문번호가 두 번 들어가고 금액 칸에 글자가 섞여도 아무도 못 막습니다), **검색**("3만원 이상 주문"을 찾으려면 매번 반복문을 새로 짜야 합니다), **동시성**(두 사람이 같은 파일에 동시에 저장하면 나중 저장이 앞사람 기록을 덮어씁니다). 데이터베이스는 이 셋을 **대신 지켜 주는** 창고입니다.

오늘은 그 창고를 직접 세웁니다. 그리고 다음다음 노트북(교안_03)에서 **둘을 한 곳에 두는 법**(관계형 DB 안의 벡터 검색)까지 갑니다.

## 1. sqlite 로 첫 표 만들기 — 파일 하나가 데이터베이스

이제 진짜 데이터베이스를 열어 봅시다. 오늘 쓸 **sqlite** 는 파이썬에 이미 들어 있어서 설치도 서버도 필요 없습니다. `output/shop.db` 라는 **파일 하나가 곧 데이터베이스**입니다.

데이터베이스에 일을 시키는 언어는 **SQL** 입니다. 하는 일은 네 가지로 모입니다.

| SQL | 하는 일 | 쇼핑몰이라면 |
|---|---|---|
| `CREATE TABLE` | 표를 만든다 | 고객 명단표를 새로 만든다 |
| `INSERT` | 행을 넣는다 | 새 고객을 등록한다 |
| `SELECT` | 행을 꺼낸다 | 서울 고객만 조회한다 |
| `UPDATE` · `DELETE` | 바꾸고 지운다 | 배송지 변경 · 주문 취소 |

적을 때의 약속 두 가지: 한 줄 주석은 `--` 입니다. 그리고 문장 끝의 `;` 는 여러 문장을 한 번에 보낼 때 구분자로 씁니다 — 이 노트북처럼 한 번에 한 문장씩 보낼 때는 생략해도 됩니다. (대소문자를 어떻게 적을지는 바로 다음 절에서 약속합니다.)

아래 준비 셀을 먼저 실행하세요. `run_sql()`(결과 없는 SQL) 과 `run_query()`(SELECT 결과를 표로) 두 개만 알면 됩니다.

In [ ]:
# [제공 코드] — sqlite 실습 준비 (내용은 이해하지 않아도 됩니다 — 실행만 하세요)
import sqlite3
from pathlib import Path

import pandas as pd

ROOT = Path(".") if Path("data").is_dir() else Path("..")
DB_PATH = ROOT / "output" / "shop.db"
DB_PATH.parent.mkdir(exist_ok=True)

_conn = None


def get_conn():
    """실습용 sqlite 연결을 하나만 만들어 계속 재사용합니다."""
    global _conn
    if _conn is None:
        _conn = sqlite3.connect(DB_PATH, isolation_level=None)  # 실행하는 즉시 저장(자동 커밋)
        _conn.execute("pragma foreign_keys = on")               # 외래키 검사를 켭니다
    return _conn


def reset_db(script=None):
    """실습 DB를 처음 상태로 되돌립니다. 언제 몇 번을 다시 실행해도 안전합니다."""
    global _conn
    if _conn is not None:
        _conn.close()
        _conn = None
    DB_PATH.unlink(missing_ok=True)
    conn = get_conn()
    if script is not None:
        conn.executescript((ROOT / "data" / script).read_text(encoding="utf-8"))
        conn.execute("pragma foreign_keys = on")
    return conn


def run_sql(sql):
    """결과가 없는 SQL(CREATE·INSERT·UPDATE·DELETE 등)을 실행합니다."""
    get_conn().execute(sql)


def run_query(sql):
    """SELECT 결과를 pandas DataFrame 으로 돌려줍니다."""
    cur = get_conn().execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[d[0] for d in cur.description])


print("sqlite 준비 완료 —", DB_PATH)

In [ ]:
# [제공 코드] — 실습 DB 전체 리셋 (언제든 다시 실행하면 빈 데이터베이스로 돌아갑니다)
reset_db()
print("빈 데이터베이스를 새로 만들었습니다.")

In [ ]:
# 첫 표 만들기 — 아직 아무 규칙도 걸지 않은 '맨몸' 표입니다.
run_sql("""
CREATE TABLE customers (
    id    integer,
    name  text,
    email text,
    city  text
)
""")

# 고객 세 명 넣기
run_sql("INSERT INTO customers (id, name, email, city) "
        "VALUES (1, '김민수', 'minsu@shop.com', '서울')")
run_sql("INSERT INTO customers (id, name, email, city) "
        "VALUES (2, '이지은', 'jieun@shop.com', '부산')")
run_sql("INSERT INTO customers (id, name, email, city) "
        "VALUES (3, '박하늘', 'haneul@shop.com', '대구')")

# 꺼내 보기
display(run_query("SELECT * FROM customers"))

표에서 **일부만** 꺼내려면 `WHERE` 뒤에 조건을 답니다. 조회는 다음 노트북(교안_02)에서 본격적으로 다루지만, 두 가지만 미리 맛봅시다.

| 쓰는 것 | 뜻 |
|---|---|
| `LIKE '김%'` | `%` 는 **아무 글자 몇 개든**(0개여도 됩니다) — '김' 으로 시작하는 모든 이름 |
| `LIKE '김_'` | `_` 는 **아무 글자 정확히 한 개** — '김' + 한 글자, 즉 **두 글자** 이름만 |
| `LIKE '김__'` | 밑줄 두 개면 '김' + 두 글자, 즉 **세 글자** 이름만 |
| `NOT 조건` | 조건을 **뒤집습니다** — `NOT city = '서울'` 은 서울이 아닌 행 |

`%` 와 `_` 의 차이는 **글자 수를 세느냐**입니다. 아래 셀에서 같은 '김' 으로 시작하는 조건인데도 결과가 달라지는 것을 보세요.

In [ ]:
# 조건 맛보기 — 지금 customers 에는 세 명(김민수·이지은·박하늘)이 들어 있습니다.
print("1) LIKE '김%' — 김으로 시작하는 모든 이름 (1행: 김민수)")
display(run_query("SELECT name, city FROM customers WHERE name LIKE '김%'"))

print("2) LIKE '김_' — 김 + 한 글자, 즉 두 글자 이름 (0행: 그런 고객이 없습니다)")
display(run_query("SELECT name, city FROM customers WHERE name LIKE '김_'"))

print("3) LIKE '김__' — 김 + 두 글자, 즉 세 글자 이름 (1행: 김민수)")
display(run_query("SELECT name, city FROM customers WHERE name LIKE '김__'"))
print("-> 2와 3은 밑줄 개수만 다른데 결과가 갈립니다. _ 는 글자 수를 정확히 셉니다.")

print()
print("4) NOT 이 조건을 뒤집습니다 — 서울이 아닌 고객 (2행)")
display(run_query("SELECT name, city FROM customers WHERE NOT city = '서울'"))

### 🖐️ 함께 따라하기 — 고객 두 명 더 넣기

`customers` 표에 아래 두 명을 넣고, 전체를 다시 조회해 5행이 나오는지 확인하세요.

| id | name | email | city |
|---|---|---|---|
| 4 | 최유진 | yujin@shop.com | 인천 |
| 5 | 정하윤 | hayoon@shop.com | 서울 |

**확인 기준**: `SELECT * FROM customers` 결과가 5행입니다.

In [ ]:
# 위 표의 두 명을 INSERT 로 넣고, 전체를 다시 조회하세요.

### ✅ 바로 확인 퀴즈

SQL 문장 `-- SELECT city FROM customers;` 를 실행하면 어떻게 될까요?

- **A.** 도시 목록이 그대로 조회됩니다
- **B.** 알 수 없는 문장이라 오류가 납니다
- **C.** 주석이라 아무 일도 없습니다
- **D.** customers 표가 통째로 지워집니다

<details><summary>정답 보기</summary>

**C** — `--` 로 시작하는 줄은 **주석**입니다. 데이터베이스는 그 줄을 읽지 않습니다.

</details>

## 2. SQL 을 적는 약속 — 키워드는 대문자, 내 이름은 소문자

SQL 은 **대소문자를 가리지 않습니다.** `select`·`SELECT`·`SeLeCt` 가 모두 같은 문장이고, 표 이름도 `customers` 와 `CUSTOMERS` 가 같은 표입니다. 그래서 **어떻게 적을지는 사람이 정하는 약속**입니다.

이 교재는 실무에서 가장 널리 쓰는 관례를 따릅니다 — **SQL 이 정해 준 문법어는 대문자, 내가 지은 이름은 소문자.** 그러면 긴 문장에서도 **어디까지가 문법이고 어디부터가 내 데이터인지** 한눈에 갈립니다.

```sql
SELECT name, city FROM customers WHERE city = '서울' ORDER BY name
```

위 문장에서 대문자만 지우면 `name, city` · `customers` · `'서울'` 만 남습니다 — 전부 **내 데이터**입니다.

| 대문자로 적는 것 | 예 |
|---|---|
| 조회·조건·정렬 | `SELECT` · `FROM` · `WHERE` · `AND` · `OR` · `NOT` · `IN` · `BETWEEN` · `LIKE` · `IS NULL` · `ORDER BY` · `LIMIT` |
| 표 만들기·고치기 | `CREATE TABLE` · `ALTER TABLE` · `ADD COLUMN` · `DROP TABLE` · `IF EXISTS` |
| 규칙(제약조건) | `PRIMARY KEY` · `FOREIGN KEY` · `REFERENCES` · `NOT NULL` · `UNIQUE` · `CHECK` · `DEFAULT` · `AUTOINCREMENT` · `STRICT` |
| 넣고·고치고·지우기 | `INSERT INTO` · `VALUES` · `UPDATE` · `SET` · `DELETE` · `RETURNING` |

| 소문자로 두는 것 | 예 |
|---|---|
| 표·열·별칭 이름 | `customers` · `customer_id` · `order_count` |
| 값(문자열) | `'서울'` · `'minsu@shop.com'` |
| 타입 | `integer` · `text` · `int` · `real` |
| 내장 함수 | `count(*)` · `sum()` · `avg()` · `date('now')` |
| sqlite 전용 지시 | `pragma foreign_keys = on` |

> 대소문자를 틀려도 **오류가 나지는 않습니다.** 이건 문법이 아니라 **읽기 좋게 하는 약속**입니다. 다만 팀이 정한 약속은 끝까지 지키는 편이 서로 읽기 좋습니다. (딱 하나 예외 — **작은따옴표 안의 값**은 진짜 데이터라서 대소문자가 달라지면 다른 값이 됩니다. `'서울'` 과 `'SEOUL'` 은 다른 도시입니다.)

### ✅ 바로 확인 퀴즈

다음 중 이 교재의 표기 약속을 **지키지 않은** 문장은 무엇일까요?

- **A.** `SELECT name FROM customers`
- **B.** `SELECT NAME FROM CUSTOMERS`
- **C.** `INSERT INTO customers (name, email) VALUES ('김민수', 'minsu@shop.com')`
- **D.** `SELECT count(*) FROM orders WHERE amount >= 30000`

<details><summary>정답 보기</summary>

**B** — `NAME` 과 `CUSTOMERS` 는 **내가 지은 이름**이라 소문자로 둡니다. 네 문장 모두 **문법으로는 멀쩡하고** 대소문자 때문에 오류가 나지도 않습니다 — 다만 약속을 따르면 문법어와 내 데이터가 한눈에 갈립니다. D 의 `count(*)` 는 내장 함수라 소문자가 맞습니다.

</details>

## 3. 데이터 타입 — 그 열이 담을 값의 종류를 미리 정한다

방금 만든 표에는 `id integer` 처럼 타입을 적긴 했지만, **규칙으로 지켜지지는 않았습니다.** 정말 그런지 확인해 봅시다.

In [ ]:
# 숫자 칸(id)에 글자를 넣어 봅니다. 막힐까요?
run_sql("INSERT INTO customers (id, name, email, city) "
        "VALUES ('일곱', '오류맨', 'bug@shop.com', '서울')")

display(run_query("SELECT * FROM customers"))
print("'일곱' 이 id 칸에 그대로 들어갔습니다 — 이제 id 로 계산도 정렬도 못 합니다.")

왜 막히지 않았을까요? **sqlite 는 기본적으로 타입을 강제하지 않습니다.** 적어 둔 타입은 '이런 값이 올 것 같다'는 힌트에 가깝습니다.

이걸 진짜 규칙으로 만들려면 표 뒤에 **`STRICT`** 를 붙입니다. 그러면 어긴 값은 저장 단계에서 거부됩니다.

**sqlite 의 타입 5가지**

| 타입 | 담는 값 | 쇼핑몰에서 예 |
|---|---|---|
| `text` | 글자 | 이름 · 이메일 · 날짜(`'2026-03-02'`) |
| `integer` · `int` | 정수 | 고객 번호 · 주문 금액 |
| `real` | 소수 | 키 · 몸무게 |
| `blob` | 이진 데이터 | 파일 원본 |
| `any` | 아무 값 | (거의 안 씁니다) |

**PostgreSQL 은 이렇게 다릅니다** — 교안_03 에서 실제로 확인합니다.

| sqlite | PostgreSQL | 메모 |
|---|---|---|
| `text` | `text` · `varchar(n)` | PostgreSQL 은 길이 제한을 타입으로 겁니다 |
| `int` | `integer` · `bigint` | |
| `real` | `numeric` · `real` | 돈은 오차 없는 `numeric` 이 안전합니다 |
| (없음) | `boolean` | sqlite 는 0·1 로 대신합니다 |
| `text` 에 ISO 문자열 | `date` · `timestamptz` | PostgreSQL 은 날짜 타입이 따로 있습니다 |
| `STRICT` 를 붙여야 검사 | 언제나 검사 | PostgreSQL 은 기본이 엄격입니다 |

In [ ]:
# STRICT 를 붙인 표는 타입을 진짜로 검사합니다.
run_sql("""
CREATE TABLE type_demo (
    id     integer,
    amount int,
    memo   text
) STRICT
""")

run_sql("INSERT INTO type_demo (id, amount, memo) VALUES (1, 29000, '정상 주문')")

try:
    run_sql("INSERT INTO type_demo (id, amount, memo) VALUES (2, '삼십이만원', '글자 금액')")
except sqlite3.IntegrityError as e:
    print("거부됨:", e)

display(run_query("SELECT * FROM type_demo"))

타입만이 아니라 **열 이름**도 표에 적어 둔 것만 쓸 수 있습니다. 표에 없는 열에 값을 넣으려 하면 저장 전에 문장 자체가 거절됩니다.

In [ ]:
# 표에 없는 열(memo2)에 값을 넣어 봅니다 — 표의 설계와 어긋나는 문장입니다.
try:
    run_sql("INSERT INTO type_demo (id, memo2) VALUES (3, '없는 열')")
except sqlite3.OperationalError as e:
    print("거부됨:", e)

### 🖐️ 함께 따라하기 — 전화번호를 어느 타입에 담을까

전화번호 `01012345678` 을 `int` 열에 담으면 무슨 일이 생기는지 직접 확인해 보세요.

1. `phone_demo` 라는 `STRICT` 표를 만드세요. 열은 `as_int int` 와 `as_text text` 둘입니다.
2. 한 행을 넣으세요 — `as_int` 에는 숫자 `01012345678` 을, `as_text` 에는 문자열 `'01012345678'` 을 넣습니다.
3. 조회해서 두 값이 어떻게 다른지 보세요.

**확인 기준**: `as_int` 쪽 값의 맨 앞 `0` 이 사라져 있습니다.

In [ ]:
# phone_demo 표를 STRICT 로 만들고 한 행을 넣은 뒤 조회하세요.

### ✅ 바로 확인 퀴즈

전화번호를 `int` 열에 담았을 때 가장 먼저 생기는 문제는 무엇일까요?

- **A.** 숫자라서 정렬이 뒤죽박죽 됩니다
- **B.** 글자 수 제한에 걸려 뒤가 잘립니다
- **C.** 저장할 때마다 미세한 오차가 생깁니다
- **D.** 맨 앞 0 이 사라져 값이 달라집니다

<details><summary>정답 보기</summary>

**D** — 숫자에게 맨 앞 `0` 은 의미가 없어서 버려집니다. **계산하지 않는 숫자**(전화번호·우편번호·상품코드)는 글자로 담습니다.

</details>

## 4. 제약조건 — 잘못된 데이터를 입구에서 막는 규칙

타입이 맞으면 다 괜찮을까요? 아닙니다. 타입은 멀쩡한데 말이 안 되는 값이 얼마든지 들어옵니다.

- 이름 칸이 텅 빈 고객이 등록됩니다.
- 같은 이메일로 두 번 가입한 고객이 생깁니다.
- 금액이 `-5000` 원인 주문이 저장됩니다.

**제약조건(constraint)** 은 "이 열엔 이런 값만 들어올 수 있다"를 표에 붙여 두는 규칙입니다. 한 번 걸어 두면 모든 입력과 수정에 자동으로 적용됩니다.

| 규칙 | 하는 일 | 쇼핑몰에서 |
|---|---|---|
| `NOT NULL` | 빈칸으로 남기지 못하게 합니다 | 고객 이름은 꼭 채워야 합니다 |
| `UNIQUE` | 같은 값이 두 번 못 들어오게 합니다 | 이메일은 고객마다 다릅니다 |
| `CHECK` | 값의 범위나 조건을 강제합니다 | 주문 금액은 0 이상만 |
| `DEFAULT` | 안 적으면 대신 채웁니다 | 가입일은 오늘 날짜로 |
| `PRIMARY KEY` | 행 하나를 콕 집는 번호표 | 고객 번호 |
| `REFERENCES` | 다른 표의 값만 허용합니다 | 주문의 고객 번호 |

앞의 `customers` 는 규칙이 없어서 이상한 값(`id = '일곱'`)까지 들어와 있습니다. **규칙을 걸어 다시 만들어** 봅시다.

In [ ]:
# 규칙 없이 만든 표와 시험용 표를 버리고, 규칙을 건 표로 새로 만듭니다.
for t in ["customers", "type_demo", "phone_demo"]:
    run_sql(f"DROP TABLE IF EXISTS {t}")

run_sql("""
CREATE TABLE customers (
    id         integer PRIMARY KEY AUTOINCREMENT,
    name       text NOT NULL,
    email      text NOT NULL UNIQUE,
    city       text,
    created_at text NOT NULL DEFAULT (date('now'))
) STRICT
""")

run_sql("INSERT INTO customers (name, email, city) VALUES ('김민수', 'minsu@shop.com', '서울')")
run_sql("INSERT INTO customers (name, email, city) VALUES ('이지은', 'jieun@shop.com', '부산')")
run_sql("INSERT INTO customers (name, email, city) VALUES ('박하늘', 'haneul@shop.com', '대구')")

display(run_query("SELECT * FROM customers"))
print("id 와 created_at 은 적지 않았는데 채워졌습니다 — "
      "integer PRIMARY KEY 와 DEFAULT 덕분입니다.")

> **`AUTOINCREMENT` 는 무엇을 더 해 주나요?** sqlite 에서는 `integer PRIMARY KEY` 만 적어도 번호가 자동으로 붙습니다. `AUTOINCREMENT` 를 덧붙이면 거기에 더해 **한 번 쓴 번호를 다시 쓰지 않는다**는 약속이 생깁니다 — 마지막 행을 지우고 새로 넣어도 지운 번호가 아니라 그다음 번호가 붙습니다. 그만큼 관리 비용이 조금 들어서, sqlite 공식 문서는 **꼭 필요한 경우가 아니면 붙이지 말라**고 권합니다. 이 교재는 번호가 재사용되지 않는 편이 읽기 쉬워 붙여 씁니다.

In [ ]:
# 규칙을 하나씩 어겨 봅니다 — 둘 다 저장되지 않아야 정상입니다.
attempts = [
    ("이름을 비움 (NOT NULL)",
     "INSERT INTO customers (name, email) VALUES (NULL, 'empty@shop.com')"),
    ("이미 있는 이메일 (UNIQUE)",
     "INSERT INTO customers (name, email) VALUES ('가짜민수', 'minsu@shop.com')"),
]

for label, sql in attempts:
    try:
        run_sql(sql)
        print(label, "-> 저장됨 (규칙이 없었다는 뜻입니다)")
    except sqlite3.IntegrityError as e:
        print(label, "-> 거부됨:", e)

print()
print("현재 고객 수:", run_query("SELECT count(*) AS n FROM customers")["n"][0])

### 🖐️ 함께 따라하기 — 규칙을 건 공급사표 만들기

쇼핑몰에 물건을 대는 **공급사** 표를 만들어 봅시다. `suppliers` 표를 `STRICT` 로 만드세요.

| 열 | 타입 | 규칙 |
|---|---|---|
| `id` | integer | 기본키, 자동 증가 |
| `name` | text | 비어 있으면 안 되고, 중복도 안 됩니다 |
| `min_order` | int | 비어 있으면 안 되고, 0 이상이어야 합니다 |
| `manager` | text | 비어 있어도 됩니다 |

만든 뒤 **한빛전자 / 100 / 김담당** 한 행을 넣고, 이어서 최소주문량이 `-10` 인 행을 넣어 보아 거부되는지 확인하세요.

**확인 기준**: 정상 행은 들어가고, 음수 최소주문량은 `CHECK constraint failed` 로 거부됩니다.

In [ ]:
# suppliers 표를 규칙과 함께 만들고, 정상 행 하나와 음수 최소주문량 한 행을 시험해 보세요.

### ✅ 바로 확인 퀴즈

주문 금액 열에 `CHECK (amount >= 0)` 을 걸어 두었습니다. 실수로 `-5000` 을 넣으면 어떻게 될까요?

- **A.** 그 줄은 거부되고 표는 그대로입니다
- **B.** 값이 자동으로 0 으로 바뀌어 저장됩니다
- **C.** 경고만 남고 그대로 저장됩니다
- **D.** 규칙이 잠시 풀리고 저장됩니다

<details><summary>정답 보기</summary>

**A** — 제약조건을 어긴 값은 고쳐지거나 경고로 끝나지 않고 **그 자리에서 거부**됩니다. 표에는 아무 변화도 남지 않습니다.

</details>

## 5. 키와 관계 — 기본키·외래키로 표를 잇는다

주문 표에 고객 **이름**을 그대로 적으면 곧 탈이 납니다. 동명이인이 생기면 누구 주문인지 가릴 수 없고, 이름이 바뀌면 그 사람 주문을 전부 찾아 고쳐야 하고, 오타 하나에 그 주문은 주인을 잃습니다.

그래서 표마다 **번호표**를 두고, 다른 표는 그 번호를 가리킵니다.

- **기본키(primary key, PK)**: 한 표 안에서 행 하나를 콕 집어내는 열. 중복도 빈값도 안 됩니다.
- **외래키(foreign key, FK)**: 다른 표의 기본키를 가리키는 열. 가리킬 값이 없으면 저장이 막힙니다.

![관계의 세 모양](images/관계의_세_모양.png)

관계는 세 모양이 있습니다. **1:1** 은 한쪽에 정확히 하나가 붙는 경우(회원 ↔ 회원상세)로 드물고, 외래키를 어느 쪽에 둬도 됩니다. 실무에서 가장 흔한 것은 **1:N** 입니다 — 한 고객이 주문을 여럿 가집니다. 이때 **외래키는 언제나 N 쪽**에 둡니다. 양쪽이 다 여럿인 **M:N** 은 관계형 표가 선 하나로 그릴 수 없어서, **중간표**를 하나 두고 1:N 두 개로 풉니다.

In [ ]:
# orders 표를 만들면서 customer_id 에 외래키를 겁니다.
run_sql("""
CREATE TABLE orders (
    id          integer PRIMARY KEY AUTOINCREMENT,
    customer_id int  NOT NULL REFERENCES customers(id),
    product     text NOT NULL,
    amount      int  NOT NULL CHECK (amount >= 0),
    ordered_at  text NOT NULL
) STRICT
""")

run_sql("INSERT INTO orders (customer_id, product, amount, ordered_at) "
        "VALUES (1, '노트북', 1290000, '2026-03-02')")

# 없는 고객 번호(99)로 주문을 넣어 봅니다.
try:
    run_sql("INSERT INTO orders (customer_id, product, amount, ordered_at) "
            "VALUES (99, '유령주문', 10000, '2026-03-03')")
except sqlite3.IntegrityError as e:
    print("거부됨:", e)

display(run_query("SELECT * FROM orders"))

> **sqlite 를 쓸 때 꼭 알아야 할 함정** — sqlite 는 외래키 검사가 **기본으로 꺼져 있습니다.** 연결할 때마다 `pragma foreign_keys = on` 을 켜야 위처럼 막힙니다. 이 노트북의 준비 셀이 이미 켜 두었습니다. 정말 그런지 아래에서 확인해 봅시다. (PostgreSQL 에는 이런 스위치가 없습니다 — 언제나 검사합니다.)

In [ ]:
# 검사를 켜지 않은 연결에서는 같은 일이 어떻게 되는지 확인합니다.
tmp = sqlite3.connect(":memory:", isolation_level=None)   # 메모리 위의 일회용 DB
tmp.execute("CREATE TABLE parent (id integer PRIMARY KEY)")
tmp.execute("CREATE TABLE child  (id integer PRIMARY KEY, pid int REFERENCES parent(id))")

print("foreign_keys 설정값:", tmp.execute("pragma foreign_keys").fetchone()[0], "(0 = 꺼짐)")
tmp.execute("INSERT INTO child (id, pid) VALUES (1, 99)")   # 부모에 99번이 없는데도?
print("넣은 결과:", tmp.execute("SELECT * FROM child").fetchall())
print("-> 검사가 꺼져 있으면 이렇게 '주인 없는 행'이 조용히 쌓입니다.")
tmp.close()

### 🖐️ 함께 따라하기 — 후기 표를 외래키로 잇기

`reviews` 표를 `STRICT` 로 만드세요.

| 열 | 타입 | 규칙 |
|---|---|---|
| `id` | integer | 기본키, 자동 증가 |
| `customer_id` | int | 비어 있으면 안 되고, `customers` 의 `id` 만 허용 |
| `product` | text | 비어 있으면 안 됩니다 |
| `rating` | int | 비어 있으면 안 되고, 1 이상 5 이하 |
| `comment` | text | 비어 있어도 됩니다 |

만든 뒤 1번 고객의 후기 한 건(**노트북 / 5점 / '화면이 밝고 가벼워요'**)을 넣고, 평점 `9` 점짜리 후기를 넣어 보아 거부되는지 확인하세요.

**확인 기준**: 정상 후기는 들어가고, 9점은 `CHECK constraint failed` 로 거부됩니다.

In [ ]:
# reviews 표를 만들고, 정상 후기 한 건과 평점 9점 한 건을 시험해 보세요.

### ✅ 바로 확인 퀴즈

학생 한 명이 과목 여럿을 듣고, 한 과목에도 학생이 여럿 있습니다. 이 관계를 표로 어떻게 만들까요?

- **A.** 학생 표에 과목 열을 여러 개 만들어 채웁니다
- **B.** 과목 표에 학생 번호 열을 하나 두고 씁니다
- **C.** 두 번호를 함께 담는 중간표를 둡니다
- **D.** 학생 표와 과목 표를 하나로 합칩니다

<details><summary>정답 보기</summary>

**C** — 양쪽이 다 여럿인 **M:N** 은 중간표로 풉니다. 중간표는 두 외래키를 함께 기본키로 삼고, 성적·학기처럼 **관계에만 있는 정보**도 담습니다.

</details>

## 6. ERD — 표를 만들기 전에 그리는 설계도

집을 짓기 전에 도면을 그리듯, 표도 그림이 먼저입니다. 그림은 몇 분이면 고치지만 이미 데이터가 쌓인 표는 고치기 어렵습니다.

**ERD(Entity Relationship Diagram)** 는 데이터 구조를 상자와 선으로 그린 설계도입니다. 들어가는 것은 딱 세 가지입니다.

- **개체(Entity)**: 저장할 대상 하나. 보통 표 한 개 (`customers`)
- **속성(Attribute)**: 그 개체가 가진 정보. 표의 열 (`name` · `email`)
- **관계(Relationship)**: 개체끼리 어떻게 이어지나 (주문은 고객에게 매달린다)

**그리는 순서는 네 걸음입니다.**

| 순서 | 하는 일 | 쇼핑몰이라면 |
|---|---|---|
| 1) 대상 찾기 | 저장할 대상을 상자로 뽑습니다 | 고객 · 주문 |
| 2) 정보 적기 | 상자마다 가진 정보를 열로 적습니다 | 이름 · 이메일 / 금액 · 날짜 |
| 3) 선 잇기 | 어느 상자끼리 이어지는지 선을 긋습니다 | 주문은 고객에게 매달립니다 |
| 4) 개수 정하기 | 선 끝에 몇 대 몇인지 표시합니다 | 한 고객에 주문 여럿 (1:N) |

![까마귀발 표기](images/까마귀발_표기.png)

**우리 쇼핑몰의 ERD** — 지금까지 만든 세 표를 한 장으로 그리면 이렇습니다. 도구는 무엇을 써도 됩니다(종이·draw.io·mermaid).

![쇼핑몰 ERD](images/erd_쇼핑몰.png)

**이름 짓는 관례** — 표 이름은 **복수형 소문자**(`customers` · `orders` · `reviews`)로, 다른 표를 가리키는 외래키는 **`가리키는표의_단수형_id`**(`customer_id` · `product_id`)로 적습니다. 이 약속만 지켜도 ERD 를 보지 않고도 어느 열이 어느 표를 가리키는지 읽힙니다.

**실무에서 자주 만나는 네 가지 모양**

| 무엇 | 모양 | 어떻게 |
|---|---|---|
| 블로그·게시판 | 1:N 이 사슬처럼 | 글은 사람에게, 댓글은 글에 매달립니다 |
| 수강신청 | M:N + 중간표 | 중간표가 학기·성적까지 가집니다 |
| 팔로우 | 한 표가 자기 자신을 | 두 열 모두 같은 `users` 를 가리킵니다 |
| 예약 | 중간표 + 자기 정보 | 사람과 회의실을 이으면서 날짜·인원도 담습니다 |

**처음 그릴 때 자주 하는 실수 네 가지**

| 이렇게 그리면 | 무슨 일이 | 이렇게 고칩니다 |
|---|---|---|
| 한 칸에 값 여러 개 | `'노트북, 마우스'` 를 한 칸에 몰아 적음 | 값 하나에 한 칸, 행을 나눕니다 |
| 같은 사실을 두 상자에 | 한쪽만 고쳐 둘이 어긋납니다 | 원본은 한 곳, 나머지는 가리킵니다 |
| M:N 을 선 하나로 | 외래키를 어디 둘지 못 정합니다 | 중간표를 둡니다 |
| 번호표 없는 상자 | 행을 하나로 집어낼 수 없습니다 | 기본키를 하나 둡니다 |

In [ ]:
# 지금 데이터베이스에 어떤 표가 있고, 각 표가 어떤 열·규칙을 가졌는지 확인합니다.
display(run_query("SELECT name FROM sqlite_master WHERE type = 'table' AND name NOT LIKE 'sqlite_%'"))

print("--- orders 표의 설계도 ---")
print(run_query("SELECT sql FROM sqlite_master WHERE name = 'orders'")["sql"][0])

### 🖐️ 함께 따라하기 — 장바구니 기능을 설계해 보기

쇼핑몰에 **장바구니** 기능을 더한다고 합시다. 한 고객이 상품 여러 개를 담고, 같은 상품이 여러 고객의 장바구니에 들어갈 수 있습니다. 담은 **수량**도 기록해야 합니다.

아래 마크다운 셀에 위의 **네 걸음** 순서대로 답을 적어 보세요. 그림 도구는 쓰지 않아도 됩니다 — 말과 표로 적으면 충분합니다.

1. 상자(표)를 몇 개 둘 것인가, 이름은 무엇인가
2. 각 상자에 어떤 열이 필요한가
3. 어느 상자끼리 잇는가
4. 몇 대 몇인가 — 그리고 외래키는 어느 상자에 두는가

*(여기에 자신의 설계를 서술하세요)*


### ✅ 바로 확인 퀴즈

위 ERD 그림에서 `customers` 와 `reviews` 를 잇는 선의 **`reviews` 쪽 끝만** 세 갈래로 갈라져 있습니다. 무슨 뜻일까요?

- **A.** 한 후기가 여러 고객에게 달립니다
- **B.** 한 고객이 후기를 여럿 남깁니다
- **C.** 고객마다 후기가 꼭 한 개입니다
- **D.** 두 표는 아직 이어지지 않았습니다

<details><summary>정답 보기</summary>

**B** — 까마귀발(세 갈래)은 그쪽이 **여럿**이라는 뜻입니다. `reviews` 쪽이 여럿이므로 한 고객이 후기를 여러 개 남깁니다. 외래키 `customer_id` 도 그 N 쪽인 `reviews` 에 있습니다.

</details>

## 7. INSERT 더 깊이 — 일부 열 · 여러 행 · 파이썬으로 적재

`INSERT INTO 표 (열들) VALUES (값들)` 이 기본형입니다. 여기서 세 가지를 더 배웁니다.

- **일부 열만** 적어도 됩니다. 안 적은 열은 `DEFAULT` 값이나 `NULL` 로 채워집니다 (단 `NOT NULL` 인데 기본값이 없으면 거부됩니다).
- **여러 행을 한 번에** 넣을 수 있습니다 — `VALUES (...), (...), (...)`.
- 넣은 뒤 **`RETURNING`** 을 붙이면 방금 만들어지거나 바뀐 값을 돌려받습니다 (`INSERT` 뿐 아니라 `UPDATE` · `DELETE` 에도 붙습니다).

그리고 실무에서 가장 많이 쓰는 모양은 **파이썬으로 여러 행을 한꺼번에 넣기**입니다. CSV 를 읽어 표에 채우는 일이 전부 이 모양입니다. 이때 값을 문장에 **글자로 이어 붙이는** 방식과 값을 **따로 건네는** 방식(파라미터 바인딩) 두 가지가 있는데, 아래 셀에서 왜 뒤엣것을 쓰는지 확인합니다.

In [ ]:
# 1) 여러 행을 한 번에
run_sql("""
INSERT INTO orders (customer_id, product, amount, ordered_at) VALUES
    (1, '마우스',   29000, '2026-03-05'),
    (2, '키보드',   89000, '2026-03-07'),
    (2, '모니터',  320000, '2026-03-11')
""")

# 2) 일부 열만 — city 를 적지 않으면 NULL 이 됩니다.
run_sql("INSERT INTO customers (name, email) VALUES ('최유진', 'yujin@shop.com')")

display(run_query("SELECT id, name, city, created_at FROM customers"))

In [ ]:
# 3) RETURNING — 방금 만들어진 id 를 돌려받습니다.
cur = get_conn().execute(
    "INSERT INTO customers (name, email, city) "
    "VALUES ('정하윤', 'hayoon@shop.com', '서울') RETURNING id, name"
)
print("새로 생긴 행:", cur.fetchone())

In [ ]:
# 3-2 UPDATE 에도 RETURNING 을 붙일 수 있습니다 — 방금 바뀐 행을 그대로 돌려받습니다.
# (UPDATE 는 'SET 열 = 새값' 으로 적고, WHERE 로 바꿀 행을 고릅니다.)
cur = get_conn().execute(
    "UPDATE customers SET city = '성남' "
    "WHERE name = '정하윤' RETURNING id, name, city"
)
print("바뀐 행:", cur.fetchone())
print("-> 몇 번 행이 어떻게 바뀌었는지 다시 조회하지 않고 바로 확인할 수 있습니다.")

In [ ]:
# 4) 파이썬으로 여러 행 넣기 — 먼저 담을 표를 만듭니다.
run_sql("""
CREATE TABLE products (
    id         integer PRIMARY KEY AUTOINCREMENT,
    name       text NOT NULL UNIQUE,
    list_price int  NOT NULL CHECK (list_price >= 0),
    category   text NOT NULL
) STRICT
""")

product_rows = [
    ("노트북", 1490000, "컴퓨터"),
    ("마우스", 35000, "주변기기"),
    ("키보드", 99000, "주변기기"),
    ("모니터", 350000, "컴퓨터"),
    ("웹캠", 79000, "주변기기"),
    ("모니터암", 59000, "주변기기"),
]

In [ ]:
# 방법 1) 값을 문장에 글자로 이어 붙이는 방식 — 만들어지는 문장을 눈으로 봅니다(아직 넣지 않습니다).
name, price, category = product_rows[0]
print(f"INSERT INTO products (name, list_price, category) "
      f"VALUES ('{name}', {price}, '{category}')")

# 그런데 값 안에 작은따옴표가 하나라도 있으면 문장이 거기서 끊깁니다.
broken_name = "모니터'암"
print(f"INSERT INTO products (name, list_price, category) "
      f"VALUES ('{broken_name}', 59000, '주변기기')")
print("-> 이름 안의 작은따옴표 때문에 값이 거기서 닫혀 버려, 뒤가 SQL 문법으로 읽힙니다.")

In [ ]:
# 방법 2) 값을 따로 건네는 방식(파라미터 바인딩) — 값 자리에 물음표만 적고 값은 튜플로 넘깁니다.
# 여러 행이면 executemany 가 그 일을 한 번에 해 줍니다. 이 방식을 기본으로 쓰세요.
get_conn().executemany(
    "INSERT INTO products (name, list_price, category) VALUES (?, ?, ?)",
    product_rows,
)

display(run_query("SELECT * FROM products ORDER BY list_price DESC"))
print("넣은 뒤 상품 수:", run_query("SELECT count(*) AS n FROM products")["n"][0])
print("-> 값에 작은따옴표가 들어 있어도 데이터베이스가 값으로만 받아들여 문장이 깨지지 않습니다.")

**값 대신 조회 결과를 넣기** — `VALUES` 자리에 `SELECT` 를 그대로 놓을 수 있습니다. 다른 표에서 뽑아낸 행을 통째로 옮겨 담을 때 씁니다.

```sql
INSERT INTO 받을표 (열1, 열2) SELECT 열A, 열B FROM 준표 WHERE 조건;
```

**열이 짝을 이루는 방식이 함정입니다.** 데이터베이스는 이름이 아니라 **적은 순서**로만 짝을 짓습니다. `열1` 에는 `열A` 가, `열2` 에는 `열B` 가 들어갑니다. 그래서 순서를 바꿔 적어도 타입만 맞으면 **오류 없이 엉뚱한 칸에 들어갑니다.** 아래에서 직접 봅시다.

In [ ]:
# 5) INSERT INTO ... SELECT — 10만원 이상 주문만 따로 보관하는 표로 옮겨 담습니다.
run_sql("""
CREATE TABLE big_orders (
    order_id    int NOT NULL,
    customer_id int NOT NULL,
    amount      int NOT NULL
) STRICT
""")

run_sql("""
INSERT INTO big_orders (order_id, customer_id, amount)
SELECT id, customer_id, amount
FROM orders
WHERE amount >= 100000
""")

display(run_query("SELECT * FROM big_orders"))

In [ ]:
# 열 순서를 뒤바꿔 넣어 봅니다 — customer_id 자리에 금액이, amount 자리에 고객 번호가 갑니다.
# 셋 다 int 라 타입 검사에 걸리지 않아 '오류 없이' 저장됩니다.
run_sql("""
INSERT INTO big_orders (order_id, customer_id, amount)
SELECT id, amount, customer_id
FROM orders
WHERE amount >= 100000
""")

display(run_query("SELECT * FROM big_orders ORDER BY order_id, amount"))
print("-> 고객 번호가 1290000 인 행이 생겼습니다. 아무도 막아 주지 않으니 순서는 직접 맞춰야 합니다.")

run_sql("DELETE FROM big_orders WHERE customer_id > 100")   # 잘못 들어간 행만 지웁니다
print("정리한 뒤 남은 행 수:", run_query("SELECT count(*) AS n FROM big_orders")["n"][0])

### ✅ 바로 확인 퀴즈

`INSERT INTO big_orders (order_id, customer_id, amount) SELECT id, amount, customer_id FROM orders` 처럼 열 순서를 바꿔 적으면 어떻게 될까요? (세 열 모두 `int` 입니다)

- **A.** 열 이름이 같으니 알아서 맞춰 들어갑니다
- **B.** 열 순서가 다르다는 오류가 납니다
- **C.** 적은 순서대로 짝지어져 엉뚱한 값이 조용히 들어갑니다
- **D.** 한 행도 들어가지 않습니다

<details><summary>정답 보기</summary>

**C** — 짝은 **이름이 아니라 적은 순서**로 지어집니다. 타입까지 맞으면 데이터베이스가 막을 방법이 없어서 오류 없이 잘못된 데이터가 쌓입니다. 넣기 전에 `SELECT` 만 따로 돌려 열 순서를 눈으로 맞춰 보세요.

</details>

### 🖐️ 함께 따라하기 — 후기 네 건을 바인딩으로 넣기

아래 `review_rows` 의 네 건을 **파라미터 바인딩**으로 `reviews` 표에 넣으세요. 튜플의 순서는 **(고객번호, 상품, 평점, 내용)** 이라 물음표 네 개가 필요합니다.

**확인 기준**: `SELECT count(*) FROM reviews` 가 **5** 입니다 (앞에서 넣은 1건 + 여기서 넣은 4건).

In [ ]:
# 넣을 후기 네 건 — (고객 id, 상품명, 별점, 내용) 순서의 튜플로 준비합니다.
# 이 순서가 곧 아래 INSERT 의 열 순서가 됩니다.
review_rows = [
    (1, "마우스", 4, "손에 잘 맞습니다"),
    (2, "키보드", 5, "타건감이 좋아요"),
    (2, "모니터", 3, "받침대가 흔들려요"),
    (1, "키보드", 2, "소리가 너무 큽니다"),
]

In [ ]:
# review_rows 를 파라미터 바인딩으로 reviews 표에 넣고, 전체 건수를 확인하세요.

### ✅ 바로 확인 퀴즈

`customers` 에 없는 고객 번호 99 로 주문을 넣으면 어떻게 될까요?

- **A.** 99번 고객이 자동으로 만들어집니다
- **B.** 주문은 저장되고 고객 자리는 비어 있습니다
- **C.** 외래키 규칙에 걸려 거부됩니다
- **D.** 경고만 나오고 저장됩니다

<details><summary>정답 보기</summary>

**C** — 외래키는 **가리킬 값이 실제로 있을 때만** 저장을 허용합니다. 그래서 '주인 없는 주문'이 생기지 않습니다. 단 sqlite 에서는 `pragma foreign_keys = on` 을 켜 두어야 합니다.

</details>

## 8. 이미 만든 표 고치기 — ALTER TABLE

쓰다 보면 열이 더 필요해집니다. 표를 다시 만들지 않고 열만 붙일 수 있습니다.

```sql
ALTER TABLE 표이름 ADD COLUMN 열이름 타입;
```

이미 들어 있는 행들의 새 열은 `NULL` 로 채워집니다. 그래서 **`NOT NULL` 인 열을 나중에 붙이려면 기본값을 함께 줘야** 합니다.

> `ALTER TABLE` 로 할 수 있는 일은 데이터베이스마다 다릅니다. sqlite 는 열을 **추가·이름 변경·삭제**까지 할 수 있지만, **이미 만든 표에 제약조건을 새로 붙이지는 못합니다.** PostgreSQL 은 그것도 됩니다 (`ALTER TABLE ... ADD CONSTRAINT`). 그래서 sqlite 에서는 **표를 만들 때 규칙을 다 정해 두는 것**이 중요합니다.

In [ ]:
# customers 에 전화번호 열을 붙입니다.
run_sql("ALTER TABLE customers ADD COLUMN phone text")
display(run_query("SELECT id, name, city, phone FROM customers"))
print("이미 있던 행들의 phone 은 NULL 입니다.")

### 🖐️ 함께 따라하기 — 상품표에 재고 열 붙이기

`products` 표에 재고를 담을 `stock` 열을 붙이세요. 타입은 `int` 이고, **비어 있으면 안 되며 기본값은 0** 입니다.

**확인 기준**: 조회했을 때 기존 상품 6개의 `stock` 이 모두 `0` 입니다.

In [ ]:
# products 에 stock 열을 붙이고 조회해 확인하세요.

### ✅ 바로 확인 퀴즈

이미 100행이 들어 있는 표에 `ALTER TABLE 표 ADD COLUMN memo text NOT NULL` 을 실행하면?

- **A.** 기존 100행의 memo 가 빈 문자열이 됩니다
- **B.** 기존 행이 모두 지워집니다
- **C.** 기본값이 없어서 실행이 거부됩니다
- **D.** 새로 넣는 행부터만 규칙이 적용됩니다

<details><summary>정답 보기</summary>

**C** — 기존 행들의 새 열은 `NULL` 이 되어야 하는데 `NOT NULL` 이 그것을 막습니다. 그래서 **`DEFAULT` 를 함께 줘야** 붙일 수 있습니다.

</details>

## 🚀 응용 클론코딩 — 장바구니를 중간표로 구현하기

§6 에서 **말로 설계한 장바구니**를 이제 진짜 표로 만듭니다. 한 고객이 상품 여럿을 담고, 한 상품도 여러 고객의 장바구니에 들어가는 **M:N** 이라 중간표가 필요합니다.

![장바구니 M:N](images/erd_장바구니_MN.png)

**할 일**

1. `cart_items` 표를 `STRICT` 로 만드세요.
   - `quantity` 는 1 이상
   - **두 외래키를 함께 묶어 기본키**로 삼으세요 (`PRIMARY KEY (customer_id, product_id)`)
   - 외래키는 열 옆이 아니라 **표 아래쪽에 따로** 적습니다 — `FOREIGN KEY (열) REFERENCES 표(열)` 두 줄로 각각 `customers(id)` · `products(id)` 만 허용하세요. 열 옆에 붙이는 `REFERENCES` 와 뜻은 같지만, 이 형태가 실무 설계 파일에서 더 흔하고 **열이 여러 개인 외래키는 이 형태로만** 쓸 수 있습니다.
2. 1번 고객의 장바구니에 1번 상품 2개, 3번 상품 1개를 담으세요. 2번 고객의 장바구니에도 1번 상품 1개를 담으세요. 값은 **파라미터 바인딩**으로 넘깁니다.
3. **같은 고객이 같은 상품을 한 번 더 담아** 보세요 — 복합 기본키가 막아야 합니다.
4. 없는 상품 번호 `99` 로 담아 보세요 — 외래키가 막아야 합니다.
5. 고객 1번의 장바구니를 조회해 2행이 나오는지 확인하세요.

**확인 기준**: `cart_items` 에 3행이 있고, 3·4 는 각각 `UNIQUE`(기본키) · `FOREIGN KEY` 오류로 거부됩니다.

In [ ]:
# 위 설계도대로 cart_items 중간표를 만들고, 데이터를 넣고, 규칙 두 가지를 시험하세요.

## 오늘 배운 것

- **sqlite** 는 파일 하나가 곧 데이터베이스입니다 — `CREATE TABLE` 로 표를 만들고 `INSERT` 로 행을 넣습니다.
- SQL 은 대소문자를 가리지 않지만, **키워드는 대문자·내가 지은 이름은 소문자**로 적어 둘을 구분합니다.
- **타입**은 값의 종류를, **제약조건**(`NOT NULL` · `UNIQUE` · `CHECK` · `DEFAULT`)은 받아 줄 값의 범위를 정합니다. sqlite 에서는 `STRICT` 를 붙여야 타입이 진짜로 검사됩니다.
- **기본키**로 행을 콕 집고 **외래키**로 표를 잇습니다. 외래키는 언제나 **N 쪽**에 둡니다. sqlite 는 `pragma foreign_keys = on` 을 켜야 검사합니다.
- **M:N** 은 중간표를 하나 둬 1:N 두 개로 풉니다.
- 전체 구조는 **ERD** 한 장으로 그려 둡니다 — 대상 찾기 → 정보 적기 → 선 잇기 → 개수 정하기.
- `INSERT`(일부 열 · 여러 행 · `RETURNING`) · `ALTER TABLE ADD COLUMN`.
- `INSERT INTO 받을표 (열들) SELECT …` 로 조회 결과를 통째로 옮겨 담습니다. 짝은 **이름이 아니라 적은 순서**로 지어지니 열 순서를 직접 맞춰야 합니다.
- 파이썬에서 값을 넣을 때는 문장에 글자로 이어 붙이지 말고 **물음표 자리에 따로 건넵니다**(파라미터 바인딩 · 여러 행이면 `executemany`).

## ⏭️ 예고 — 다음 노트북: SQL 조회와 결합

표는 만들었고 데이터도 넣었습니다. `LIKE` 와 `NOT` 으로 조건도 살짝 맛봤죠. 다음 시간엔 그 안에서 **원하는 것만 꺼내는 법**을 제대로 배웁니다. 조건으로 거르고(`WHERE`), 줄 세우고(`ORDER BY`), 묶어 계산하고(`GROUP BY`), 흩어진 표를 이어(`JOIN`) 리포트 한 장을 만듭니다.